# Investigate and setup paths

In [1]:
from pathlib import Path

In [2]:
print(Path.cwd())

/Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks


In [3]:
for path in sorted(Path.cwd().iterdir()): 
    print(" -", path)

 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/.DS_Store
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/.ipynb_checkpoints
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/data
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/data_collection.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/evals
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/evals.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/courses/deeplearning_posttraining/notebooks/mlops_prep_llm_training_serving.ipynb
 - /Users/unmeshmali/Downloads/Unmesh/deepmlhub/projects/llm-retard-lab/cour

In [4]:
config_path = Path("../configs/config.yaml")

# Read the configs

In [5]:
import yaml

In [6]:
with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

In [7]:
print(type(config))

<class 'dict'>


In [8]:
print(config)

{'model': {'name': 'Qwen/Qwen2.5-1.5B'}, 'data': {'path': 'data/01_cold_start_cot_sft/data.jsonl'}, 'output': {'directory': 'models/adapter_experiment'}, 'peft': {'method': 'lora', 'r': 16, 'alpha': 32, 'dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']}, 'training': {'num_train_epochs': 3, 'batch_size': 4, 'gradient_accumulation_steps': 4, 'learning_rate': 0.1002, 'max_length': 1024, 'seed': 42}}


In [9]:
from pprint import pprint

In [10]:
print("Model config:")
pprint(config["model"], sort_dicts=False)

Model config:
{'name': 'Qwen/Qwen2.5-1.5B'}


## Reading configs in variables

In [11]:
model_name = config["model"]["name"]
data_path = config["data"]["path"]
peft_method = config["peft"]["method"]
learning_rate = config["training"]["learning_rate"]

In [12]:
print("Model:", model_name)
print("Data:", data_path)
print("PEFT method:", peft_method)
print("Learning rate:", learning_rate)

Model: Qwen/Qwen2.5-1.5B
Data: data/01_cold_start_cot_sft/data.jsonl
PEFT method: lora
Learning rate: 0.1002


In [13]:
values_to_check = {
    "model_name": model_name,
    "data_path": data_path,
    "peft_method": peft_method,
    "learning_rate": learning_rate,
    "target_modules": config["peft"]["target_modules"],
}

In [14]:
for name, value in values_to_check.items():
    print(f"{name}: {value!r} | type={type(value).__name__}")

model_name: 'Qwen/Qwen2.5-1.5B' | type=str
data_path: 'data/01_cold_start_cot_sft/data.jsonl' | type=str
peft_method: 'lora' | type=str
learning_rate: 0.1002 | type=float
target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'] | type=list


## Validation configuations 

In [15]:
assert isinstance(config, dict)
assert config["model"]["name"]
assert config["peft"]["method"] in {"lora", "adalora", "ia3"}
assert config["training"]["learning_rate"] > 0
assert isinstance(config["peft"]["target_modules"], list)


# Connecting the above config to the existing TrainConfig class

## data and path shenanigans

In [18]:
from dataclasses import asdict
from pathlib import Path
from llm_training.config import TrainConfig

In [17]:
import sys
sys.path.append('../src/')

In [19]:
project_root = config_path.parent.parent

In [20]:
print(project_root)

..


In [21]:
data_path = project_root / config["data"]["path"]
output_dir = project_root / config["output"]["directory"]

In [23]:
print(output_dir)

../models/adapter_experiment


In [24]:
print("Data path:", data_path)
print("Exists:", data_path.exists())
print("Output directory:", output_dir)

Data path: ../data/01_cold_start_cot_sft/data.jsonl
Exists: True
Output directory: ../models/adapter_experiment


## Building TrainConfig class instance

In [25]:
train_config = TrainConfig(
    model_name=config["model"]["name"],
    data_path=data_path,
    output_dir=output_dir,
    lora_r=config["peft"]["r"],
    lora_alpha=config["peft"]["alpha"],
    lora_dropout=config["peft"]["dropout"],
    lora_target_modules=tuple(config["peft"]["target_modules"]),
    num_train_epochs=config["training"]["num_train_epochs"],
    per_device_train_batch_size=config["training"]["batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    learning_rate=config["training"]["learning_rate"],
    max_length=config["training"]["max_length"],
    seed=config["training"]["seed"],
)

In [26]:
for name, value in asdict(train_config).items():
    print(f"{name}: {value!r}")

model_name: 'Qwen/Qwen2.5-1.5B'
data_path: PosixPath('../data/01_cold_start_cot_sft/data.jsonl')
output_dir: PosixPath('../models/adapter_experiment')
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target_modules: ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')
num_train_epochs: 3
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 0.1002
max_length: 1024
warmup_ratio: 0.03
lr_scheduler_type: 'cosine'
logging_steps: 10
eval_steps: 50
save_steps: 200
save_total_limit: 2
max_steps: None
fp16: False
bf16: True
seed: 42
report_to: 'none'
